In [ ]:
import torch

from test_attn_order_eval import AttnOrderEval   # signal-agnostic ranking eval over a [T, L] per-step score + order


class ConfidenceOrderEval(AttnOrderEval):

    def _confidence(self, logits):   # logits [T, L, V] -> top-1 softmax prob (confidence) [T, L]
        logits = logits.to(torch.float64)                            # match the float64 softmax convention; chunk over T if memory-bound
        lse = torch.logsumexp(logits, dim=-1)                        # [T, L]  log-partition (full vocab scan)
        top1 = logits.max(dim=-1).values                            # [T, L]  largest logit
        return (top1 - lse).exp()                                    # [T, L]  p1 = exp(l1 - logZ)
    # end

    def __init__(self, logits, order):                              # logits [T, L, V], order [T] long
        assert logits.dim() == 3, "logits.dim(): {} == 3 false".format(logits.dim())

        score = self._confidence(logits)                            # [T, L]  per-step, per-position confidence
        super().__init__(score, order)                              # reuse geometry + recall_at_h / pr_auc / ndcg_at_h
    # end

    def get_confidence(self):
        return self.attn   # the base stores the ranking score here (generic [T, L] score slot)
    # end


if __name__ == "__main__":
    torch.manual_seed(0)
    T, L, V = 64, 64, 50
    order = torch.randperm(L)

    step_of = torch.full((L,), -1, dtype=torch.long)
    step_of[order] = torch.arange(T)
    gap = step_of.view(1, L) - torch.arange(T).view(T, 1)          # [T, L]

    # perfect: peak token 0 higher (=> higher confidence) for sooner-unmasked candidates
    peak = torch.where(gap > 0, 5.0 / gap.clamp_min(1).float(), torch.zeros(1))   # [T, L]
    logits_perfect = torch.zeros(T, L, V)
    logits_perfect[..., 0] = peak
    logits_random = torch.randn(T, L, V)

    def summ(x):
        return "{:.3f} (n={})".format(x.nanmean().item(), int((~x.isnan()).sum()))
    # end

    for name, lg in [("perfect", logits_perfect), ("random ", logits_random)]:
        ev = ConfidenceOrderEval(lg, order)
        print("{}  recall@5={}  pr_auc@5={}  ndcg@10={}".format(
            name, summ(ev.recall_at_h(5)), summ(ev.pr_auc(5)), summ(ev.ndcg_at_h(10))))
    # end